In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# 1. Cargar el dataset original (asegúrate de que el archivo bank-full.csv está en la misma carpeta)
print("Cargando datos...")
df = pd.read_csv('bank-full.csv', sep=';')

# 2. Dividir: 90% para modelado, 10% para simulación de streaming
df_90, df_10 = train_test_split(df, test_size=0.10, random_state=42)

# Guardar el 10% en un CSV para que el Productor lo use después
df_10.to_csv('dataset_10_streaming.csv', index=False)
print("✅ Guardado el 10% de los datos para la simulación (dataset_10_streaming.csv).")

# 3. Preparar los datos del 90% para entrenar
X = df_90.drop('y', axis=1)
y = df_90['y'].apply(lambda x: 1 if x == 'yes' else 0)

# Sacamos Train (70%) y un temporal (30%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
# Del temporal sacamos Validación (15%) y Test (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# 4. Crear el preprocesador para convertir textos a números y escalar
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Unimos el preprocesador y el algoritmo en un "Pipeline"
modelo = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 5. Entrenar el modelo
print("Entrenando el modelo (esto puede tardar unos segundos)...")
modelo.fit(X_train, y_train)

# 6. Guardar el modelo entrenado en tu disco duro
joblib.dump(modelo, 'modelo_bankfull.pkl')
print("✅ ¡Parte 1 Completada! Modelo guardado como 'modelo_bankfull.pkl'.")